<a href="https://colab.research.google.com/github/DhariniVenki/CS769-Bias-Content-Moderation/blob/main/Content_Moderation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data extraction and preprocessing

Builds every dataset the project needs from **ETHOS** (Mollas et al., 2020) and **Civil Comments**
(Borkan et al., 2019), both human-labelled social-media / news comments. **ToxicCommons** (Arnett et al.,
2024) is optional and off by default: its texts are mostly OCR'd books and newspapers, and its labels
come from an LLM. A review of the training split found most of its "toxic" labels on ads, church
history and war news, and 12.6% of its posts not in English.

| Output | Used for |
|---|---|
| `train.csv`, `val.csv` | Fine-tuning M0 → M2 (binary toxic label) |
| `test.csv` | Strictly held-out split: all bias testing and counterfactual perturbation |
| `verification_300.csv` | The 300 posts for LLM tagging and verification: 100 human-only, 100 Gemini-only, 100 both (200 + 200 with 100 overlap) |
| `biascompare_gender_test.jsonl` | Held-out gender posts in the `biascompare` input format |
| `all_posts.csv` | Every cleaned post with its tags and split, for analysis |

Steps: load → clean → deduplicate → label → tag gender and disability mentions →
stratified split → sample the verification set → save. Every step is seeded.

Run the cells top to bottom. The offline test cell checks the functions on small
made-up data before the real download.

## Setup

Clones the repo and installs `biascompare` (used for its gender lexicon) and Hugging Face
`datasets` (used to download ToxicCommons-balanced). Set
`SAVE_TO_DRIVE = True` to keep the outputs after the Colab runtime resets.

In [ ]:
import os, sys

REPO_URL = "https://github.com/DhariniVenki/CS769-Bias-Content-Moderation.git"
REPO_DIR = "/content/CS769-Bias-Content-Moderation"
SAVE_TO_DRIVE = False

if "google.colab" in sys.modules:
    if os.path.isdir(REPO_DIR):
        !git -C {REPO_DIR} pull -q
    else:
        !git clone -q {REPO_URL} {REPO_DIR}
    %pip install -q -e {REPO_DIR} datasets
    os.chdir(REPO_DIR)
    if SAVE_TO_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")

sys.path.insert(0, os.path.abspath("src"))

## Configuration

All the knobs in one place. Raise `CC_SAMPLE_SIZE` if the held-out split comes up short of axis posts
(the build cell warns you). `USE_TOXICCOMMONS = True` adds ToxicCommons back, with the stricter rules
below.

In [ ]:
from pathlib import Path

SEED = 0

OUT_DIR = Path("/content/drive/MyDrive/CS769/data/processed" if SAVE_TO_DRIVE else "data/processed")

ETHOS_URLS = {
    "binary": "https://raw.githubusercontent.com/intelligence-csd-auth-gr/Ethos-Hate-Speech-Dataset/master/ethos/ethos_data/Ethos_Dataset_Binary.csv",
    "multilabel": "https://raw.githubusercontent.com/intelligence-csd-auth-gr/Ethos-Hate-Speech-Dataset/master/ethos/ethos_data/Ethos_Dataset_Multi_Label.csv",
}
ETHOS_HATE_THRESHOLD = 0.5    # isHate is the share of annotators who said "hate"; ≥ 0.5 = hate (ETHOS paper)
ETHOS_TARGET_THRESHOLD = 0.5  # same rule for the multilabel target columns (gender, disability, ...)

CC_DATASET = "google/civil_comments"  # ~1.8M English news-site comments, CC0; toxicity = share of raters
USE_CIVIL_COMMENTS = True
CC_TOXIC_THRESHOLD = 0.5       # toxic if at least half the raters said toxic (Borkan et al., 2019)
CC_SAMPLE_SIZE = 60_000        # comments kept, half toxic and half clean
CC_TARGET_DISABILITY = True    # also keep disability-mentioning comments (the scarce axis), balanced:
CC_TARGET_CLEAN_PER_TOXIC = 1  #   every toxic one plus this many clean ones per toxic one

USE_TOXICCOMMONS = False       # off: OCR'd books/newspapers with LLM-made labels (see the intro)
TC_DATASET = "tcapelle/ToxicCommons-balanced"  # toxic/clean-balanced ToxicCommons, English + Spanish
TC_SAMPLE_SIZE = 50_000        # English ToxicCommons posts kept, half toxic and half clean
TC_MILD_LABEL = 1              # "mild" tier (summed score 4–6, or 3 on one axis): 1 = toxic, 0 = clean, None drops it
TC_AXIS_TOXIC_MIN = None       # None: a single axis scoring >= this no longer makes a post toxic (it labelled
                               # ads and history toxic); set 2 to restore the old rule
TC_TARGETED_AXES = ("ability", "gender_sex")  # keep every English row scoring ≥ 1 on these, on top of the sample
TC_MIN_ENGLISH_SHARE = 0.15    # ToxicCommons only: common English words must be >= 15% of the words

MAX_CLEAN_PER_TOXIC = 3        # cap on clean posts per toxic post, per source, in train/val only

MIN_WORDS = 4
MAX_CHARS = 2000               # about the 512 tokens the classifiers read; longer texts are skipped, not cut
MIN_ALPHA_RATIO = 0.6          # drops OCR junk in the public-domain scans

SPLIT_FRACTIONS = {"train": 0.7, "val": 0.1, "test": 0.2}

VERIFY_N = 300                 # 100 human-only + 100 Gemini-only + 100 both
VERIFY_AXIS_SHARE = {"gender": 0.5, "disability": 0.5}  # unused share is filled from the other axis

## Loading

**Civil Comments** (`google/civil_comments`) has about 1.8 million comments from news sites, each with
`toxicity` = the share of crowd raters who called it toxic. A comment is toxic if `toxicity ≥ 0.5`. The
loader keeps comments of at most `MAX_CHARS` characters, samples `CC_SAMPLE_SIZE` of them (half toxic, half
clean), and adds every toxic comment that mentions disability plus as many clean ones, so the scarce axis
is well covered and balanced. Its `identity_attack` score is kept: the training script uses it to skip
counterfactual pairs on comments that attack a group.

**ETHOS** has two semicolon-separated CSVs. The binary file has all 998 comments; the
multilabel file is the 433 hate comments again, with target columns (`gender`,
`disability`, ...). The two are joined on the comment text, so each comment appears once.

**ToxicCommons** scores each text 0–3 on race/origin, gender/sex, religion, ability and
violence. The raw release is almost all clean (one run gave ~44 toxic posts in 400k
rows), so this uses `tcapelle/ToxicCommons-balanced`, a derivative balanced between
toxic and clean, with columns `text`, `scores` (the five 0–3 scores) and one flag per
axis. It has no overall toxic column, so the label comes from the paper's tiers on the
summed scores: `toxic` (≥ 7) is 1, `clean` is 0, and `mild` follows `TC_MILD_LABEL`
(toxic by default: the paper calls that tier "mild toxicity"). On top of that, a row
with any single axis scored `TC_AXIS_TOXIC_MIN` or more is toxic, which catches abuse
aimed at one group that the sum misses.

It mixes English and Spanish, and the classifiers and lexicons are English, so
`looks_english` keeps rows where common English words outnumber common Spanish ones.
Then up to `TC_SAMPLE_SIZE` rows are sampled, half toxic and half clean, and every
English row scoring 1 or more on a `TC_TARGETED_AXES` axis (ability, gender/sex) is
added on top, so the bias axes get as many targeted posts as the dataset has.

The loader prints a funnel: rows and toxic rows left after each filter. If toxic rows
vanish at one step, that filter is the one to loosen.

In [ ]:
import ast
import io
import re
import urllib.request

import numpy as np
import pandas as pd

TC_AXES = ("race_origin", "gender_sex", "religion", "ability", "violence")
ETHOS_TARGETS = ("violence", "directed_vs_generalized", "gender", "race",
                 "national_origin", "disability", "religion", "sexual_orientation")


def open_text(source):
    """A text stream for a local path, an http(s) URL, or an already-open file-like."""
    if hasattr(source, "read"):
        return source
    source = str(source)
    if source.startswith(("http://", "https://")):
        return io.TextIOWrapper(urllib.request.urlopen(source), encoding="utf-8", newline="")
    return open(source, encoding="utf-8", newline="")


def load_ethos(binary_source, multilabel_source):
    binary = pd.read_csv(open_text(binary_source), sep=";")
    multilabel = pd.read_csv(open_text(multilabel_source), sep=";")
    for frame, name in ((binary, "binary"), (multilabel, "multilabel")):
        if "comment" not in frame.columns:
            raise ValueError(f"ETHOS {name} CSV has no 'comment' column: {list(frame.columns)}")

    targets = multilabel.drop_duplicates("comment").set_index("comment")
    targets = (targets[list(ETHOS_TARGETS)] >= ETHOS_TARGET_THRESHOLD).astype(int)
    targets.columns = [f"ethos_target_{c}" for c in targets.columns]

    frame = binary.join(targets, on="comment")
    target_cols = list(targets.columns)
    frame[target_cols] = frame[target_cols].fillna(0).astype(int)
    frame["label"] = (frame["isHate"] >= ETHOS_HATE_THRESHOLD).astype(int)
    return pd.DataFrame({
        "id": [f"ethos-{i:04d}" for i in range(len(frame))],
        "source": "ethos",
        "platform": "youtube/reddit",
        "raw_text": frame["comment"],
        "label": frame["label"],
        "toxicity_tier": frame["label"].map({1: "toxic", 0: "clean"}),
        "hate_score": frame["isHate"],
        **{c: frame[c] for c in target_cols},
    })


def toxiccommons_tier(scores):
    """The ToxicCommons paper's tiers on the summed 0–3 scores."""
    total = sum(scores)
    if total >= 7:
        return "toxic"
    if 4 <= total <= 6 or (total == 3 and max(scores) == 3):
        return "mild"
    return "clean"


EN_STOPWORDS = frozenset(
    "the and of to in that is was he she it for with as his her on be at by i you this had "
    "not are but from or have an they which were would their there been has will".split()
)
ES_STOPWORDS = frozenset(
    "el la de que y en los las del se por un una con es para al lo como su más pero sus "
    "le ya fue este esta son entre cuando muy sin sobre también".split()
)
_LATIN_WORD_RE = re.compile(r"[^\W\d_]+")


def looks_english(text, min_share=TC_MIN_ENGLISH_SHARE):
    """True when common English words outnumber common Spanish ones and make up at least
    `min_share` of the words. Used on ToxicCommons only: at 5% it let through German, Dutch, Polish and
    Latin OCR, so the default is 15%. Short social-media insults have few stopwords, so ETHOS and Civil
    Comments are not filtered this way."""
    words = [w.lower() for w in _LATIN_WORD_RE.findall(str(text))]
    if not words:
        return False
    en = sum(w in EN_STOPWORDS for w in words)
    es = sum(w in ES_STOPWORDS for w in words)
    return en > es and en / len(words) >= min_share


def _parse_scores(value):
    """Five int scores from a list, array or "[0, 1, ...]" string, or None if malformed."""
    try:
        if isinstance(value, str):
            value = ast.literal_eval(value)
        scores = [int(x) for x in value]
    except (TypeError, ValueError, SyntaxError):
        return None
    return scores if len(scores) == len(TC_AXES) else None


def tc_labels(scores, tiers, mild_label=TC_MILD_LABEL, axis_toxic_min=TC_AXIS_TOXIC_MIN):
    """1/0 per row from the tiers, then 1 wherever one axis reaches `axis_toxic_min`.
    Rows left without a label (mild, when `mild_label` is None) are NaN."""
    labels = tiers.map({"toxic": 1, "clean": 0, "mild": mild_label}).astype(float)
    if axis_toxic_min is not None:
        labels[scores.map(max) >= axis_toxic_min] = 1
    return labels


def load_toxiccommons_balanced(frame, sample_size=TC_SAMPLE_SIZE, max_chars=MAX_CHARS,
                               mild_label=TC_MILD_LABEL, axis_toxic_min=TC_AXIS_TOXIC_MIN,
                               targeted_axes=TC_TARGETED_AXES, seed=SEED, verbose=True):
    """`frame` is ToxicCommons-balanced as a DataFrame. Keeps English rows of at most
    `max_chars` characters with five valid scores and a label, then takes a sample of up to
    `sample_size` rows (half toxic, half clean) plus every row scoring ≥ 1 on a
    `targeted_axes` axis. Ids come from the row position in the full dataset, so they stay
    the same whatever the sample size."""
    missing = {"text", "scores"} - set(frame.columns)
    if missing:
        raise ValueError(f"ToxicCommons-balanced is missing column(s) {sorted(missing)}: {list(frame.columns)}")
    frame = frame[frame["text"].notna()]
    scores = frame["scores"].map(_parse_scores)
    frame, scores = frame[scores.notna()], scores[scores.notna()]
    tiers = scores.map(toxiccommons_tier)
    labels = tc_labels(scores, tiers, mild_label, axis_toxic_min)

    funnel = []
    def step(name, keep):
        nonlocal frame
        frame = frame[keep.loc[frame.index]]
        funnel.append((name, len(frame), int((labels.loc[frame.index] == 1).sum())))

    step("valid scores", pd.Series(True, index=frame.index))
    step("labelled", labels.notna())
    step(f"≤ {max_chars} chars", frame["text"].str.len() <= max_chars)
    step("English", frame["text"].map(looks_english))
    if verbose:
        print("ToxicCommons funnel (rows, toxic):")
        for name, n_rows, n_toxic in funnel:
            print(f"  {name:>16}: {n_rows:>9,} {n_toxic:>9,}")
        print("  tiers left:", tiers.loc[frame.index].value_counts().to_dict())

    kept = labels.loc[frame.index].astype(int)
    per_label = sample_size // 2
    picked = set()
    for _, group in kept.groupby(kept, sort=True):
        picked.update(group.sample(n=min(len(group), per_label), random_state=seed).index)
    axis_idx = [TC_AXES.index(a) for a in targeted_axes]
    targeted = scores.loc[frame.index].map(lambda s: any(s[i] >= 1 for i in axis_idx))
    picked.update(frame.index[targeted.to_numpy()])
    index = pd.Index(sorted(picked))
    if verbose:
        print(f"  sampled {len(index):,} rows ({int(targeted.sum()):,} targeted on {list(targeted_axes)}), "
              f"{int(kept.loc[index].sum()):,} toxic")

    scores = scores.loc[index]
    return pd.DataFrame({
        "id": [f"tc-{i:07d}" for i in index],
        "source": "toxiccommons",
        "platform": "public-domain",
        "raw_text": frame.loc[index, "text"].to_numpy(),
        "label": kept.loc[index].to_numpy(),
        "toxicity_tier": tiers.loc[index].to_numpy(),
        **{f"tc_{axis}": scores.map(lambda s, i=i: s[i]).to_numpy() for i, axis in enumerate(TC_AXES)},
    })


def load_civil_comments(frame, sample_size=CC_SAMPLE_SIZE, threshold=CC_TOXIC_THRESHOLD, max_chars=MAX_CHARS,
                        targeted=None, clean_per_toxic=CC_TARGET_CLEAN_PER_TOXIC, seed=SEED, verbose=True):
    """`frame` is Civil Comments as a DataFrame (`text`, `toxicity`, ...). Keeps comments of at most
    `max_chars` characters, labels toxic where `toxicity >= threshold`, and samples up to `sample_size`
    (half toxic, half clean). `targeted(text) -> bool` marks extra rows to keep (disability mentions):
    all toxic ones plus `clean_per_toxic` clean ones per toxic one. Ids come from the row position in the
    full dataset, so they stay the same whatever the sample size."""
    missing = {"text", "toxicity"} - set(frame.columns)
    if missing:
        raise ValueError(f"Civil Comments is missing column(s) {sorted(missing)}: {list(frame.columns)}")
    n_all = len(frame)
    frame = frame[frame["text"].notna() & frame["toxicity"].notna()]
    frame = frame[frame["text"].str.len() <= max_chars]
    labels = (frame["toxicity"] >= threshold).astype(int)
    rng_state = seed
    picked = set()
    for _, group in labels.groupby(labels, sort=True):
        picked.update(group.sample(n=min(len(group), sample_size // 2), random_state=rng_state).index)
    n_targeted = 0
    if targeted is not None:
        hit = frame["text"].map(targeted).astype(bool)
        toxic = frame.index[(hit & (labels == 1)).to_numpy()]
        clean = frame.index[(hit & (labels == 0)).to_numpy()]
        n_clean = min(len(clean), int(len(toxic) * clean_per_toxic))
        extra = set(toxic) | set(pd.Series(clean).sample(n=n_clean, random_state=seed).tolist() if n_clean else [])
        n_targeted = len(extra - picked)
        picked |= extra
    index = pd.Index(sorted(picked))
    if verbose:
        print(f"Civil Comments: {n_all:,} rows, {len(frame):,} of at most {max_chars} chars "
              f"({int(labels.sum()):,} toxic); kept {len(index):,} ({n_targeted:,} extra targeted), "
              f"{int(labels.loc[index].sum()):,} toxic")
    rows = frame.loc[index]
    tier = np.where(rows["toxicity"] >= threshold, "toxic", "clean")
    out = {
        "id": [f"cc-{i:07d}" for i in index],
        "source": "civil_comments",
        "platform": "news comments",
        "raw_text": rows["text"].to_numpy(),
        "label": labels.loc[index].to_numpy(),
        "toxicity_tier": tier,
        "cc_toxicity": rows["toxicity"].to_numpy(),
    }
    for column in ("identity_attack", "insult", "threat"):
        if column in rows.columns:
            out[f"cc_{column}"] = rows[column].to_numpy()
    return pd.DataFrame(out)


## Cleaning and deduplication

- Unescapes HTML, normalizes Unicode (NFKC), drops zero-width characters.
- Removes URLs, replaces `@handles` with `@user`, rejoins OCR line-break hyphens
  (`moder-\nation` → `moderation`), collapses whitespace.
- Drops texts with fewer than `MIN_WORDS` words, more than `MAX_CHARS` characters, or
  less than `MIN_ALPHA_RATIO` letters (OCR junk).
- Deduplicates on a normalized key (lowercase, letters and digits only) **across both
  datasets before splitting**, so the same text can never land in train and test.

In [ ]:
import html
import re
import unicodedata

URL_RE = re.compile(r"(https?://\S+|www\.\S+)", re.IGNORECASE)
HANDLE_RE = re.compile(r"(?<!\w)@\w+")
HYPHEN_BREAK_RE = re.compile(r"(\w)-\s*\n\s*(\w)")
ZERO_WIDTH_RE = re.compile(r"[​-‏⁠﻿]")
SPACE_RE = re.compile(r"\s+")
DEDUP_RE = re.compile(r"[^a-z0-9]+")


def clean_text(text):
    text = html.unescape(str(text))
    text = unicodedata.normalize("NFKC", text)
    text = ZERO_WIDTH_RE.sub("", text)
    text = HYPHEN_BREAK_RE.sub(r"\1\2", text)
    text = URL_RE.sub(" ", text)
    text = HANDLE_RE.sub("@user", text)
    return SPACE_RE.sub(" ", text).strip()


def alpha_ratio(text):
    visible = [c for c in text if not c.isspace()]
    return sum(c.isalpha() for c in visible) / len(visible) if visible else 0.0


def dedup_key(text):
    return DEDUP_RE.sub(" ", text.lower()).strip()


def clean_frame(frame, min_words=MIN_WORDS, max_chars=MAX_CHARS, min_alpha=MIN_ALPHA_RATIO):
    """Returns (cleaned frame, counts of rows dropped per reason)."""
    frame = frame.copy()
    frame["text"] = frame["raw_text"].map(clean_text)
    n_words = frame["text"].str.split().str.len()
    reasons = {
        "too_short": n_words < min_words,
        "too_long": frame["text"].str.len() > max_chars,
        "low_alpha": frame["text"].map(alpha_ratio) < min_alpha,
    }
    drop = pd.Series(False, index=frame.index)
    dropped = {}
    for reason, mask in reasons.items():
        dropped[reason] = int((mask & ~drop).sum())
        drop |= mask
    frame = frame[~drop].copy()

    frame["dedup_key"] = frame["text"].map(dedup_key)
    before = len(frame)
    # Keep the toxic copy when duplicates disagree, so a clean duplicate can't hide hate.
    frame = (frame.sort_values(["label", "id"], ascending=[False, True])
                  .drop_duplicates("dedup_key")
                  .sort_values("id"))
    dropped["duplicate"] = before - len(frame)
    return frame.reset_index(drop=True), dropped

## Tagging gender and disability mentions

This finds which posts *mention* each axis, so they can be swapped later. It is a
lexicon pre-filter, not the final tagging: the proposal's 8B LLM decides the actual
swap words and control synonyms on the verification set.

- **Gender** uses the `biascompare` gender lexicon, the same one the pipeline swaps with. A possessive
  `'s` is stripped first, so "women's sports" counts as a mention of women.
  `gender_group` is the majority gender referenced (male/female), or empty on a tie.
- **Disability** uses the term list below, based on the categories in Hutchinson et al.
  (2020): sensory, mobility, cognitive, mental health and chronic illness, including
  slurs, since those are exactly the posts a moderator must catch. Words that are
  mostly non-disability in old newspapers (depression, invalid, mute) are left out.
  Extend it as needed.

The dataset annotations are kept alongside: ETHOS `ethos_target_gender` and
`ethos_target_disability` (the hate is *aimed at* that group), and ToxicCommons
`tc_gender_sex` and `tc_ability` scores.

In [ ]:
import json

import importlib.resources

_LEXICON = importlib.resources.files("biascompare") / "resources" / "gender_lexicon.json"
_GENDER_PAIRS = [(m.lower(), f.lower()) for m, f in json.loads(_LEXICON.read_text(encoding="utf-8"))["pairs"]]
MALE_TERMS = frozenset(m for m, _ in _GENDER_PAIRS)
FEMALE_TERMS = frozenset(f for _, f in _GENDER_PAIRS)
WORD_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")
POSSESSIVE_RE = re.compile(r"'s$")  # "women's" -> "women"; the curly ’ already splits the word

DISABILITY_TERMS = (
    # sensory
    "blind", "deaf", "hard of hearing", "visually impaired", "hearing impaired",
    # mobility / physical
    "wheelchair", "paralyzed", "paralysed", "paraplegic", "quadriplegic",
    "amputee", "crippled", "cripple", "handicapped", "cerebral palsy", "dwarf", "dwarfism",
    # cognitive / developmental
    "autistic", "autism", "asperger", "aspergers", "down syndrome", "down's syndrome",
    "downs syndrome", "dyslexic", "dyslexia", "adhd", "intellectual disability",
    "learning disability", "retard", "retarded", "retards", "mongoloid", "special needs",
    # mental health
    "mentally ill", "mental illness", "schizophrenic", "schizophrenia", "bipolar",
    "psychotic", "ptsd", "ocd", "clinical depression", "anxiety disorder", "lunatic", "psycho",
    # chronic illness / general
    "epileptic", "epilepsy", "chronic illness", "disabled", "disability", "disabilities",
    "deformed",
)
DISABILITY_RE = re.compile(
    r"\b(" + "|".join(re.escape(t) for t in sorted(DISABILITY_TERMS, key=len, reverse=True))
    .replace(r"\ ", r"\s+").replace("'", "['’]?") + r")\b",
    re.IGNORECASE,
)


def gender_mentions(text):
    words = [POSSESSIVE_RE.sub("", w.lower()) for w in WORD_RE.findall(text)]
    male = [w for w in words if w in MALE_TERMS]
    female = [w for w in words if w in FEMALE_TERMS]
    if len(male) == len(female):
        group = None
    else:
        group = "male" if len(male) > len(female) else "female"
    return male + female, group


def disability_mentions(text):
    return [SPACE_RE.sub(" ", m.group(0).lower()) for m in DISABILITY_RE.finditer(text)]


def tag_axes(frame):
    frame = frame.copy()
    gender = frame["text"].map(gender_mentions)
    frame["gender_terms"] = gender.map(lambda g: "|".join(g[0]))
    frame["gender_group"] = gender.map(lambda g: g[1])
    frame["mentions_gender"] = gender.map(lambda g: bool(g[0]))
    frame["disability_terms"] = frame["text"].map(lambda t: "|".join(disability_mentions(t)))
    frame["mentions_disability"] = frame["disability_terms"] != ""
    frame["axis"] = [
        "both" if g and d else "gender" if g else "disability" if d else "none"
        for g, d in zip(frame["mentions_gender"], frame["mentions_disability"])
    ]
    return frame

## Splits and the verification set

**Split.** Stratified on source × label × axis, so train, val and test each get the
same mix of toxic and clean posts, and of gender, disability and neither. Test is held
out: no fine-tuning ever sees it, and all counterfactual testing runs on it.

**Balancing.** After the split, `balance_train_val` caps clean posts at
`MAX_CLEAN_PER_TOXIC` per toxic post within each source's train and val rows, so
fine-tuning isn't swamped by clean text. Test is never rebalanced: it keeps every post,
so the bias tests see benign mentions of gender and disability too.

**Verification set.** 300 test posts that mention an axis, split between gender and
disability by `VERIFY_AXIS_SHARE`. Within each axis the sampler takes posts round-robin
across strata (male/female or disability × toxic/clean × source), so no stratum
dominates. Then each post is assigned to `human`, `gemini` or `both` in turn, giving
100 each: 200 for the human pass and 200 for Gemini, with 100 checked by both for the
agreement measure. Posts tagged `both` axes count toward disability, the rarer one.

In [ ]:
import numpy as np


def stratified_split(frame, fractions=SPLIT_FRACTIONS, by=("source", "label", "axis"), seed=SEED):
    names = list(fractions)
    split = pd.Series(index=frame.index, dtype=object)
    rng = np.random.default_rng(seed)
    for _, group in frame.groupby(list(by), sort=True):
        index = rng.permutation(group.index.to_numpy())
        counts = _largest_remainder([fractions[n] * len(index) for n in names])
        start = 0
        for name, count in zip(names, counts):
            split.loc[index[start:start + count]] = name
            start += count
    return split


def balance_train_val(frame, max_clean_per_toxic=MAX_CLEAN_PER_TOXIC, seed=SEED):
    drop = []
    fit_rows = frame[frame["split"].isin(["train", "val"])]
    for _, group in fit_rows.groupby(["split", "source"], sort=True):
        clean = group[group["label"] == 0]
        cap = max_clean_per_toxic * int((group["label"] == 1).sum())
        if len(clean) > cap:
            drop.extend(clean.sample(n=len(clean) - cap, random_state=seed).index)
    return frame.drop(index=drop).reset_index(drop=True)


def _largest_remainder(quotas):
    """Rounds quotas to integers that sum to round(sum(quotas))."""
    floors = [int(q) for q in quotas]
    remainder = round(sum(quotas)) - sum(floors)
    order = sorted(range(len(quotas)), key=lambda i: quotas[i] - floors[i], reverse=True)
    for i in order[:remainder]:
        floors[i] += 1
    return floors


def round_robin_sample(frame, by, n, seed=SEED):
    """Up to `n` rows, taken one per stratum in turn (each stratum shuffled), so small
    strata are represented as fully as large ones allow."""
    rng = np.random.default_rng(seed)
    queues = [list(rng.permutation(g.index.to_numpy())) for _, g in frame.groupby(by, sort=True)]
    picked = []
    while len(picked) < n and any(queues):
        for queue in queues:
            if queue and len(picked) < n:
                picked.append(queue.pop(0))
    return frame.loc[picked]


def verification_axis(row):
    return "disability" if row["mentions_disability"] else "gender"


def build_verification_set(test, n=VERIFY_N, axis_share=VERIFY_AXIS_SHARE, seed=SEED, warn=True):
    pool = test[test["axis"] != "none"].copy()
    pool["verify_axis"] = pool.apply(verification_axis, axis=1)
    pool["stratum_group"] = np.where(pool["verify_axis"] == "gender", pool["gender_group"].fillna("tie"), "disability")
    by = ["stratum_group", "label", "source"]

    targets = {axis: int(round(n * share)) for axis, share in axis_share.items()}
    available = pool["verify_axis"].value_counts().to_dict()
    # Hand any shortfall in one axis to the other.
    for axis, other in (("disability", "gender"), ("gender", "disability")):
        shortfall = targets[axis] - available.get(axis, 0)
        if shortfall > 0:
            targets[axis] -= shortfall
            targets[other] += shortfall

    parts = [round_robin_sample(pool[pool["verify_axis"] == axis], by, targets[axis], seed)
             for axis in ("gender", "disability")]
    chosen = pd.concat(parts)
    if warn and len(chosen) < n:
        print(f"WARNING: only {len(chosen)} axis posts in test, wanted {n}. Raise TC_SAMPLE_SIZE.")

    # Interleave the axes so the assignment cycle spreads both evenly over annotators.
    order = round_robin_sample(chosen, ["verify_axis"], len(chosen), seed)
    roles = ("both", "human", "gemini")
    order = order.assign(annotators=[roles[i % 3] for i in range(len(order))])
    for column in ("llm_swap_words", "llm_control_synonyms", "human_verdict", "gemini_verdict", "notes"):
        order[column] = ""
    return order.drop(columns=["stratum_group"]).reset_index(drop=True)

## Saving

In [ ]:
import json

POST_COLUMNS = ["id", "source", "platform", "text", "label", "toxicity_tier", "axis",
                "gender_group", "gender_terms", "disability_terms"]


def to_biascompare_jsonl(frame, path):
    """Gender posts in the `biascompare.load_posts` format. Not filtered for synonym
    controls: the LLM-tagged synonyms will replace the POC's 49-word list."""
    rows = frame[frame["gender_group"].isin(["male", "female"])]
    with open(path, "w", encoding="utf-8") as handle:
        for row in rows.itertuples():
            handle.write(json.dumps({
                "id": row.id, "platform": row.platform, "text": row.text,
                "intended_toxicity": row.toxicity_tier, "gender": row.gender_group,
            }, ensure_ascii=False) + "\n")
    return len(rows)


def save_outputs(frame, verification, out_dir=OUT_DIR):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    frame.drop(columns=["raw_text"]).to_csv(out_dir / "all_posts.csv", index=False)
    for split in SPLIT_FRACTIONS:
        frame.loc[frame["split"] == split, POST_COLUMNS].to_csv(out_dir / f"{split}.csv", index=False)
    verification.drop(columns=["raw_text"], errors="ignore").to_csv(out_dir / "verification_300.csv", index=False)
    n_gender = to_biascompare_jsonl(frame[frame["split"] == "test"], out_dir / "biascompare_gender_test.jsonl")
    print(f"Saved to {out_dir.resolve()} ({n_gender} posts in biascompare_gender_test.jsonl)")


def build_dataset(*sources, seed=SEED, warn=True):
    """The whole pipeline after loading, on any loaded sources (ETHOS, Civil Comments, ToxicCommons).
    Returns (posts with a split column, verification set, drop counts)."""
    combined = pd.concat([s for s in sources if s is not None], ignore_index=True)
    cleaned, dropped = clean_frame(combined)
    tagged = tag_axes(cleaned)
    tagged["split"] = stratified_split(tagged, seed=seed)
    before = len(tagged)
    tagged = balance_train_val(tagged, seed=seed)
    dropped["train_val_clean_downsampled"] = before - len(tagged)
    verification = build_verification_set(tagged[tagged["split"] == "test"], seed=seed, warn=warn)
    return tagged, verification, dropped

## Offline tests

Small in-memory CSVs, no network. One line per test; raises if any fail.

In [ ]:
import tempfile
from collections import Counter

ETHOS_BINARY_FIXTURE = """comment;isHate
She is a terrible driver and her opinions are garbage;1.0
He is the worst coach and his team knows it;0.8
The weather is really nice today in town;0.0
My sister said she loved the movie a lot;0.0
Blind people should not be allowed to vote at all;0.9
Check this out https://t.co/abc @someone it's great stuff;0.0
too short;0.0
The weather is REALLY nice today, in town!;1.0
"""

ETHOS_MULTILABEL_FIXTURE = """comment;violence;directed_vs_generalized;gender;race;national_origin;disability;religion;sexual_orientation
She is a terrible driver and her opinions are garbage;0;0.2;0.833333;0;0;0;0;0
Blind people should not be allowed to vote at all;0;0;0;0;0;1;0;0
"""


def _tc_frame(rows):
    """rows: (text, scores) tuples, shaped like ToxicCommons-balanced."""
    return pd.DataFrame(rows, columns=["text", "scores"])


def _ethos():
    return load_ethos(io.StringIO(ETHOS_BINARY_FIXTURE), io.StringIO(ETHOS_MULTILABEL_FIXTURE))


def test_ethos_labels_and_targets_join():
    ethos = _ethos()
    assert len(ethos) == 8
    assert ethos["label"].tolist() == [1, 1, 0, 0, 1, 0, 0, 1]
    assert ethos["ethos_target_gender"].tolist() == [1, 0, 0, 0, 0, 0, 0, 0]
    assert ethos["ethos_target_disability"].tolist() == [0, 0, 0, 0, 1, 0, 0, 0]


def test_ethos_rejects_wrong_csv():
    try:
        load_ethos(io.StringIO("text;label\nhi;0\n"), io.StringIO(ETHOS_MULTILABEL_FIXTURE))
    except ValueError:
        return
    raise AssertionError("expected ValueError")


def test_toxiccommons_tiers():
    assert toxiccommons_tier([0, 0, 0, 0, 0]) == "clean"
    assert toxiccommons_tier([1, 1, 1, 0, 0]) == "clean"  # sum 3, spread out
    assert toxiccommons_tier([0, 3, 0, 0, 0]) == "mild"   # sum 3, one axis
    assert toxiccommons_tier([2, 2, 0, 0, 0]) == "mild"
    assert toxiccommons_tier([3, 3, 1, 0, 0]) == "toxic"


def test_looks_english():
    assert looks_english("He said that the house was on the hill")
    assert not looks_english("El hombre dijo que la casa de su madre es muy grande")
    assert not looks_english("12 34 -- ##")


def test_parse_scores():
    assert _parse_scores("[0, 1, 2, 3, 0]") == [0, 1, 2, 3, 0]
    assert _parse_scores((0, 0, 0, 0, 1)) == [0, 0, 0, 0, 1]
    assert _parse_scores([0, 1]) is None and _parse_scores(None) is None and _parse_scores("junk") is None


def test_toxiccommons_balanced_loader():
    rows = [("His wife was a fine nurse in the town", [0, 0, 0, 0, 0]),
            ("They will kill every last one of those women", [2, 3, 0, 0, 3]),
            ("The man said the cripple was a burden to the town", [0, 0, 0, 3, 3]),  # sum 6 → mild
            ("La mujer dijo que el hombre es muy malo", [0, 3, 0, 3, 3]),              # Spanish
            ("x" * (MAX_CHARS + 1) + " the", [0, 0, 0, 0, 0]),                         # too long
            ("The man said the cripple was a burden", [0, 0, 0, 3])]                   # 4 scores
    strict = dict(mild_label=None, axis_toxic_min=None, targeted_axes=(), verbose=False)
    tc = load_toxiccommons_balanced(_tc_frame(rows), **strict)
    assert tc["id"].tolist() == ["tc-0000000", "tc-0000001"]
    assert tc["label"].tolist() == [0, 1]
    assert tc["toxicity_tier"].tolist() == ["clean", "toxic"]
    assert tc["tc_gender_sex"].tolist() == [0, 3]
    mild_as_toxic = load_toxiccommons_balanced(_tc_frame(rows), **{**strict, "mild_label": 1})
    assert mild_as_toxic["label"].tolist() == [0, 1, 1]


def test_tc_labels_axis_rule():
    scores = pd.Series([[0, 2, 0, 0, 0], [1, 1, 0, 0, 0], [0, 0, 0, 0, 3]])
    tiers = scores.map(toxiccommons_tier)  # clean, clean, mild
    assert tc_labels(scores, tiers, mild_label=None, axis_toxic_min=None).tolist()[:2] == [0, 0]
    assert tc_labels(scores, tiers, mild_label=None, axis_toxic_min=2).tolist() == [1, 0, 1]


def test_toxiccommons_targeted_rows_are_always_kept():
    rows = [(f"He said the row {i} was fine", [0, 0, 0, 0, 0]) for i in range(100)]
    rows += [(f"The blind man said the row {i} was fine", [0, 0, 0, 1, 0]) for i in range(30)]
    tc = load_toxiccommons_balanced(_tc_frame(rows), sample_size=10, verbose=False)
    assert (tc["tc_ability"] == 1).sum() == 30   # all targeted rows, though the sample holds 10


def test_toxiccommons_balanced_sample_is_even_and_seeded():
    rows = [(f"He said the row {i} was fine", [3, 3, 1, 0, 0] if i % 4 == 0 else [0, 0, 0, 0, 0])
            for i in range(400)]
    first = load_toxiccommons_balanced(_tc_frame(rows), sample_size=100, targeted_axes=("violence",), verbose=False)
    assert Counter(first["label"]) == {0: 50, 1: 50}
    again = load_toxiccommons_balanced(_tc_frame(rows), sample_size=100, targeted_axes=("violence",), verbose=False)
    assert first["id"].tolist() == again["id"].tolist()


def test_toxiccommons_balanced_rejects_missing_columns():
    try:
        load_toxiccommons_balanced(pd.DataFrame({"text": ["hi"]}), verbose=False)
    except ValueError:
        return
    raise AssertionError("expected ValueError")


def test_clean_text():
    assert clean_text("Hi&amp;bye https://x.co/a @bob  moder-\n ation​") == "Hi&bye @user moderation"


def test_clean_frame_drops_and_dedups_keeping_toxic_copy():
    cleaned, dropped = clean_frame(_ethos())
    assert dropped["too_short"] == 1 and dropped["duplicate"] == 1
    weather = cleaned[cleaned["text"].str.lower().str.contains("weather")]
    assert len(weather) == 1 and weather["label"].item() == 1


def test_tagging():
    assert gender_mentions("He said his sister left") == (["he", "his", "sister"], "male")
    assert gender_mentions("He told her") == (["he", "her"], None)
    assert disability_mentions("My Down’s  syndrome cousin is not a retard, he's autistic") == [
        "down’s syndrome", "retard", "autistic"]
    assert disability_mentions("The blinds were closed") == []
    tagged = tag_axes(clean_frame(_ethos())[0])
    assert dict(zip(tagged["id"], tagged["axis"]))["ethos-0004"] == "disability"


def test_balance_train_val_leaves_test_alone():
    frame = pd.DataFrame({
        "split": ["train"] * 10 + ["test"] * 10,
        "source": "toxiccommons",
        "label": [1, 1] + [0] * 8 + [1] + [0] * 9,
    })
    out = balance_train_val(frame, max_clean_per_toxic=2)
    assert Counter(out.loc[out["split"] == "train", "label"]) == {1: 2, 0: 4}
    assert Counter(out.loc[out["split"] == "test", "label"]) == {1: 1, 0: 9}


def test_split_is_stratified_seeded_and_disjoint():
    frame = pd.DataFrame({"source": ["a"] * 80 + ["b"] * 40, "label": [0, 1] * 60, "axis": "none"})
    split = stratified_split(frame, seed=1)
    assert split.equals(stratified_split(frame, seed=1))
    assert Counter(split) == {"train": 84, "val": 12, "test": 24}
    assert Counter(split[(frame["source"] == "b") & (frame["label"] == 1)]) == {"train": 14, "val": 2, "test": 4}


def _synthetic_test_pool(n_gender, n_disability):
    rows = []
    for i in range(n_gender):
        rows.append({"id": f"g{i}", "source": "ethos", "label": i % 2, "axis": "gender",
                     "mentions_disability": False, "gender_group": ["male", "female"][i % 2]})
    for i in range(n_disability):
        rows.append({"id": f"d{i}", "source": "toxiccommons", "label": i % 2, "axis": "disability",
                     "mentions_disability": True, "gender_group": None})
    rows.append({"id": "n0", "source": "ethos", "label": 0, "axis": "none",
                 "mentions_disability": False, "gender_group": None})
    return pd.DataFrame(rows)


def test_verification_set_sizes_and_overlap():
    ver = build_verification_set(_synthetic_test_pool(400, 400))
    assert len(ver) == 300 and ver["id"].is_unique and "n0" not in set(ver["id"])
    assert Counter(ver["annotators"]) == {"both": 100, "human": 100, "gemini": 100}
    assert Counter(ver["verify_axis"]) == {"gender": 150, "disability": 150}
    human = ver["annotators"].isin(["human", "both"]).sum()
    gemini = ver["annotators"].isin(["gemini", "both"]).sum()
    assert human == gemini == 200
    both = ver[ver["annotators"] == "both"]
    assert Counter(both["verify_axis"]) == {"gender": 50, "disability": 50}
    gender = ver[ver["verify_axis"] == "gender"]
    assert Counter(gender["gender_group"]) == {"male": 75, "female": 75}


def test_verification_set_fills_disability_shortfall_with_gender():
    ver = build_verification_set(_synthetic_test_pool(400, 40))
    assert Counter(ver["verify_axis"]) == {"gender": 260, "disability": 40}


def test_end_to_end_on_fixtures():
    tc_rows = [(f"{w} said {p} kept waiting in the hall, row {i}", [0, 0, 0, 0, 0])
               for i in range(30) for w, p in (("My brother", "he"), ("My sister", "she"))]
    tc_rows += [(f"The blind man was cast out of the town by force, row {i}", [1, 0, 0, 3, 3]) for i in range(5)]
    tc_rows += [(f"Kill all of those disabled women in the town now, row {i}", [0, 3, 0, 3, 3]) for i in range(10)]
    tc = load_toxiccommons_balanced(_tc_frame(tc_rows), verbose=False)
    posts, ver, dropped = build_dataset(_ethos(), tc, warn=False)
    assert posts["dedup_key"].is_unique
    assert set(posts["split"]) == {"train", "val", "test"}
    assert set(ver["id"]) <= set(posts.loc[posts["split"] == "test", "id"])
    with tempfile.TemporaryDirectory() as tmp:
        save_outputs(posts, ver, tmp)
        assert sorted(p.name for p in Path(tmp).iterdir()) == [
            "all_posts.csv", "biascompare_gender_test.jsonl", "test.csv", "train.csv",
            "val.csv", "verification_300.csv"]
        from biascompare import load_posts
        assert len(load_posts(Path(tmp) / "biascompare_gender_test.jsonl")) > 0


def _cc_frame(rows):
    """rows: (text, toxicity) tuples, shaped like Civil Comments."""
    return pd.DataFrame(rows, columns=["text", "toxicity"]).assign(identity_attack=0.0, insult=0.0, threat=0.0)


def test_civil_comments_labels_and_balanced_sample():
    rows = [(f"what a great article number {i}", 0.0) for i in range(300)]
    rows += [(f"you are a complete idiot number {i}", 0.8) for i in range(100)]
    rows += [("x" * (MAX_CHARS + 1), 0.9)]
    cc = load_civil_comments(_cc_frame(rows), sample_size=100, verbose=False)
    assert Counter(cc["label"]) == {0: 50, 1: 50}
    assert set(cc["source"]) == {"civil_comments"} and cc["id"].str.startswith("cc-").all()
    assert (cc.loc[cc["label"] == 1, "cc_toxicity"] >= CC_TOXIC_THRESHOLD).all()
    again = load_civil_comments(_cc_frame(rows), sample_size=100, verbose=False)
    assert cc["id"].tolist() == again["id"].tolist()


def test_civil_comments_threshold_is_half_the_raters():
    cc = load_civil_comments(_cc_frame([("a b c d", 0.49), ("e f g h", 0.5)]), sample_size=10, verbose=False)
    assert dict(zip(cc["raw_text"], cc["label"])) == {"a b c d": 0, "e f g h": 1}


def test_civil_comments_targeted_rows_are_kept_and_balanced():
    rows = [(f"nothing to see number {i}", 0.0) for i in range(200)]
    rows += [(f"my deaf cousin is great number {i}", 0.0) for i in range(50)]
    rows += [(f"deaf people are idiots number {i}", 0.9) for i in range(10)]
    cc = load_civil_comments(_cc_frame(rows), sample_size=20, verbose=False,
                             targeted=lambda t: bool(DISABILITY_RE.search(t)))
    deaf = cc[cc["raw_text"].str.contains("deaf")]
    assert (deaf["label"] == 1).sum() == 10          # every targeted toxic row
    assert 10 <= (deaf["label"] == 0).sum() <= 20     # matched clean ones (plus any the random sample drew)


def test_civil_comments_rejects_missing_columns():
    try:
        load_civil_comments(pd.DataFrame({"text": ["hi"]}), verbose=False)
    except ValueError:
        return
    raise AssertionError("expected ValueError")


def test_possessive_gender_words_are_tagged():
    assert gender_mentions("You should know women's sports are a joke") == (["women"], "female")
    assert gender_mentions("lived in your mother's basement")[1] == "female"
    assert gender_mentions("the Asian woman’s smarts")[1] == "female"     # curly apostrophe


def test_toxiccommons_english_filter_is_strict():
    assert not looks_english("Zooals we allemaal wel weten, is de bevolking in die provincie over het geheel Roomsch")
    assert not looks_english("für die Mädchen, für die Wittwen, für die Frauen überhaupt in Norwegen 26,98 Jahre")
    assert looks_english("He said that the house was on the hill and it was by the river")


def test_toxiccommons_axis_rule_is_off_by_default():
    scores = pd.Series([[0, 2, 0, 0, 0]])
    assert tc_labels(scores, scores.map(toxiccommons_tier)).tolist() == [0]


def test_end_to_end_with_civil_comments():
    rows = [(f"{w} said {p} loved the new park downtown, number {i}", 0.0)
            for i in range(30) for w, p in (("My brother", "he"), ("My sister", "she"))]
    rows += [(f"Shut up you idiot, {p} is a moron, number {i}", 0.9) for i in range(30) for p in ("he", "she")]
    rows += [(f"The blind man was turned away by the bus driver, number {i}", 0.0) for i in range(10)]
    rows += [(f"Disabled people are a waste of space, number {i}", 0.9) for i in range(10)]
    cc = load_civil_comments(_cc_frame(rows), sample_size=200, verbose=False,
                             targeted=lambda t: bool(DISABILITY_RE.search(t)))
    posts, ver, _ = build_dataset(_ethos(), cc, warn=False)
    assert set(posts["source"]) == {"ethos", "civil_comments"}
    assert posts["dedup_key"].is_unique and set(posts["split"]) == {"train", "val", "test"}
    assert {"cc_identity_attack", "cc_toxicity"} <= set(posts.columns)
    assert set(ver["id"]) <= set(posts.loc[posts["split"] == "test", "id"])


def run_tests():
    tests = [(name, fn) for name, fn in list(globals().items()) if name.startswith("test_") and callable(fn)]
    failures = 0
    for name, fn in tests:
        try:
            fn()
            print(f"PASS {name}")
        except Exception as exc:
            failures += 1
            print(f"FAIL {name}: {type(exc).__name__}: {exc}")
    print(f"\n{len(tests) - failures}/{len(tests)} passed")
    if failures:
        raise AssertionError(f"{failures} test(s) failed")


run_tests()

## Load the real data

ETHOS is small. Civil Comments is a ~400 MB download (1.8M rows), loaded into memory once and then
sampled. ToxicCommons (only with `USE_TOXICCOMMONS = True`) is ~820 MB more. Use a high-RAM runtime if
it runs out of memory.

In [ ]:
ethos = load_ethos(ETHOS_URLS["binary"], ETHOS_URLS["multilabel"])
print(f"ETHOS: {len(ethos)} comments, {ethos['label'].sum()} hate")

from datasets import load_dataset

civil = toxiccommons = None
if USE_CIVIL_COMMENTS:
    cc_raw = load_dataset(CC_DATASET, split="train").to_pandas()
    civil = load_civil_comments(
        cc_raw, targeted=(lambda t: bool(DISABILITY_RE.search(t))) if CC_TARGET_DISABILITY else None)
    del cc_raw
if USE_TOXICCOMMONS:
    tc_raw = load_dataset(TC_DATASET, split="train").to_pandas()
    print(f"ToxicCommons-balanced: {len(tc_raw)} rows, columns {list(tc_raw.columns)}")
    toxiccommons = load_toxiccommons_balanced(tc_raw)
    del tc_raw
    print(f"ToxicCommons: {len(toxiccommons)} rows kept, {toxiccommons['label'].sum()} toxic")


## Build and save

In [ ]:
posts, verification, dropped = build_dataset(ethos, civil, toxiccommons)
print("Dropped:", dropped)

summary = posts.groupby(["split", "source", "axis"]).agg(n=("id", "size"), toxic=("label", "sum")).unstack("axis", fill_value=0)
display(summary)
display(pd.crosstab(verification["verify_axis"], verification["annotators"], margins=True))

save_outputs(posts, verification)